# Imports

In [36]:
import pickle
import pickletools
import datetime
from eolearn.core import EOPatch
from pyproj import Proj, transform

import numpy as np
import pandas as pd

import geopandas as gpd
import rasterio as rio
from rasterio import features

# Data Pre-processing

## Understanding Data

### Helper functions

In [3]:
def formatCordinates(cordinate): 
    # Define the UTM projection (for example, UTM Zone 33N)
    utm_proj = Proj(proj="utm", zone=33, datum="WGS84")
    
    # Define the WGS84 (lat, lon) projection
    latlon_proj = Proj(proj="latlong", datum="WGS84")
    
    # Convert UTM to Latitude/Longitude
    longitude, latitude = transform(utm_proj, latlon_proj, cordinate[0], cordinate[1])
    
    print(f"Latitude = {latitude}, Longitude = {longitude}")

### Germany - 2018 - 243N

In [19]:
user_path = 'C:/Users/Sai Suhaas' 
base_path_2018_242N = user_path + '/Desktop/Dissertation/Farmland-Classification-using-Remote-Sensing/Dataset/s2-utm-33N-18E-242N-2018'
train_geojson_path  = user_path + '/Desktop/Dissertation/Farmland-Classification-using-Remote-Sensing/Dataset/br-18E-242N-crop-labels-train-2018.geojson'

#### Meta info

In [5]:
# Replace 'filename.pkl' with the path to your pickle file
path = base_path_2018_242N + '/meta_info.pkl'

with open(path, 'rb') as file:
    pickled_data = pickletools.dis(file)

    0: \x80 PROTO      3
    2: c    GLOBAL     'eolearn.core.eodata _FeatureDict'
   36: q    BINPUT     0
   38: )    EMPTY_TUPLE
   39: \x81 NEWOBJ
   40: q    BINPUT     1
   42: (    MARK
   43: X        BINUNICODE 'size_x'
   54: q        BINPUT     2
   56: M        BININT2    2400
   59: X        BINUNICODE 'size_y'
   70: q        BINPUT     3
   72: M        BININT2    2400
   75: X        BINUNICODE 'time_interval'
   93: q        BINPUT     4
   95: c        GLOBAL     'datetime datetime'
  114: q        BINPUT     5
  116: C        SHORT_BINBYTES b'\x07\xe2\x01\x01\x00\x00\x00\x00\x00\x00'
  128: q        BINPUT     6
  130: \x85     TUPLE1
  131: q        BINPUT     7
  133: R        REDUCE
  134: q        BINPUT     8
  136: h        BINGET     5
  138: C        SHORT_BINBYTES b'\x07\xe2\x0c\x1f\x17;;\x00\x00\x00'
  150: q        BINPUT     9
  152: \x85     TUPLE1
  153: q        BINPUT     10
  155: R        REDUCE
  156: q        BINPUT     11
  158: \x86     TUPLE2
 

In [6]:
with open(path, 'rb') as file:
    data = file.read()  # Read the first 100 bytes
    print(data)

b'\x80\x03ceolearn.core.eodata\n_FeatureDict\nq\x00)\x81q\x01(X\x06\x00\x00\x00size_xq\x02M`\tX\x06\x00\x00\x00size_yq\x03M`\tX\r\x00\x00\x00time_intervalq\x04cdatetime\ndatetime\nq\x05C\n\x07\xe2\x01\x01\x00\x00\x00\x00\x00\x00q\x06\x85q\x07Rq\x08h\x05C\n\x07\xe2\x0c\x1f\x17;;\x00\x00\x00q\t\x85q\nRq\x0b\x86q\x0cX\x05\x00\x00\x00maxccq\rK\x01X\x0f\x00\x00\x00time_differenceq\x0ecdatetime\ntimedelta\nq\x0fK\x00M \x1cK\x00\x87q\x10Rq\x11u}q\x12(X\x0c\x00\x00\x00feature_typeq\x13ceolearn.core.constants\nFeatureType\nq\x14X\t\x00\x00\x00meta_infoq\x15\x85q\x16Rq\x17X\x04\x00\x00\x00ndimq\x18NX\t\x00\x00\x00is_vectorq\x19\x89ub.'


#### Timestamps

In [7]:
path = base_path_2018_242N + '/timestamp.pkl'

try:
    # Replace 'filename.pkl' with the path to your pickle file
    with open(path, 'rb') as file:
        data = pickle.load(file)

        readable_dates = [dt.strftime("%Y-%m-%d %H:%M:%S") for dt in data]
    
        # Now 'data' contains the contents of the pickle file
        #print(readable_dates)

        print(f"Start date: {min(readable_dates)}\nEnd Date: {max(readable_dates)}")
        print(f"No of timestamps: {len(readable_dates)}")
except Exception as e:
    print(f"An error occurred: {e}")


Start date: 2018-01-02 10:24:20
End Date: 2018-12-30 10:16:06
No of timestamps: 144


#### Area Cordinates 

In [8]:
path = base_path_2018_242N + '/bbox.pkl'

try:
    # Replace 'filename.pkl' with the path to your pickle file
    with open(path, 'rb') as file:
        data = list(pickle.load(file))

    
        # Now 'data' contains the contents of the pickle file
        print("Start point: ")
        formatCordinates(data[:2])
        print("End point: ")
        formatCordinates(data[2:])
except Exception as e:
    print(f"An error occurred: {e}")


Start point: 
Latitude = 52.417989999999996, Longitude = 14.000120000000003
End point: 
Latitude = 52.63620000888608, Longitude = 14.349811619358587


C:\Users\Sai Suhaas\AppData\Local\Temp\ipykernel_9680\2877271017.py:9: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  longitude, latitude = transform(utm_proj, latlon_proj, cordinate[0], cordinate[1])
C:\Users\Sai Suhaas\AppData\Local\Temp\ipykernel_9680\2877271017.py:9: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  longitude, latitude = transform(utm_proj, latlon_proj, cordinate[0], cordinate[1])


#### Load Data

In [9]:
path = base_path_2018_242N + '/BANDS.npy'

# Load the .npy file
data_2018_242N = np.load(path)

In [10]:
# Now 'data' holds the contents of the file
print(data_2018_242N.shape)

data_2018_242N_start_date = data_2018_242N[0, :, :, 1]

(144, 2400, 2400, 12)


#### Load mask

In [15]:
path = base_path_2018_242N + '/SCL.npy'
cloud_mask_2018_242N = np.load(path)

In [16]:
print(cloud_mask_2018_242N.shape)

print(np.unique(cloud_mask_2018_242N[:][:][:]))

(144, 2400, 2400, 1)
[ 0  2  3  4  5  6  7  8  9 10 11]


#### Load the json and labels

In [58]:
min_area = 500 # in meters/square
labels = gpd.read_file(train_geojson_path)

with open((base_path_2018_242N + '/bbox.pkl'), 'rb') as f:
            bbox = pickle.load(f)
            crs = str(bbox.crs)
            minx, miny, maxx, maxy = bbox.min_x, bbox.min_y, bbox.max_x, bbox.max_y

# project to same coordinate reference system (crs) as the imagery
labels = labels.to_crs(crs)

mask = labels.geometry.area > min_area
print(f"ignoring {(~mask).sum()}/{len(mask)} fields with area < {min_area}m2")
labels = labels.loc[mask]

#self.bands = np.load(os.path.join(rootpath, "data", "BANDS.npy"))
#self.clp = np.load(os.path.join(rootpath, "data", "CLP.npy"))
# bands = np.concatenate([bands, clp], axis=-1) # concat cloud probability
_, width, height, _ = data_2018_242N.shape

transform = rio.transform.from_bounds(minx, miny, maxx, maxy, width, height)

fid_mask = features.rasterize(zip(labels.geometry, labels.fid), 
                              all_touched=True,
                              transform=transform, 
                              out_shape=(width, height),
                              default_value=-1,
                              nodata=-1)

assert len(np.unique(fid_mask)) > 0, f"vectorized fid mask contains no fields. " \
                                     f"Does the label geojson {train_geojson_path} cover the region defined by {base_path_2018_242N}?"

crop_mask = features.rasterize(zip(labels.geometry, labels.crop_id),
                               all_touched=True,
                               transform=transform,
                               out_shape=(width, height),
                               default_value=-1,
                               nodata=-1)

assert len(np.unique(crop_mask)) > 0, f"vectorized fid mask contains no fields. " \
                                      f"Does the label geojson {train_geojson_path} cover the region defined by {base_path_2018_242N}?"

#Replace 0 with -1 to indicate clear difference between the no data areaa
#To be noted while defining loss function to make discourage class imbalance
crop_mask = np.where(crop_mask == 0, -1, crop_mask)

ignoring 13/2534 fields with area < 500m2


In [49]:
crop_mask.shape

(2400, 2400)

In [59]:
np.unique(crop_mask, return_counts=True)

(array([-1,  1,  2,  3,  4,  5,  6,  7,  8,  9]),
 array([2797961,  623835,  453365,  279643,   48665,  544897,  507156,
          16917,  262077,  225484]))